# Perceptron to backprop: hands-on (optional)
**LLMs From Basics · Lesson H1.1**

You'll replay forty years of AI history in four small exercises:
1. Rosenblatt's **perceptron** (1958): predict, then learn with his update rule.
2. Watch it **converge** on data a straight line can separate.
3. Watch it **fail on XOR**, and check that *no* line can do better than 3 of 4.
4. Build a tiny **two-layer network** trained with **backpropagation** (1986) that solves XOR.

Each exercise has a cell with a `# TODO`, followed by a **check** cell with `assert`s. Run it; a ✓ means success. Solutions are at the very bottom. Try first: *struggling to recall is what makes it stick.*

Only `numpy` and `matplotlib` are needed (both pre-installed on Colab).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

# The four logic functions as data: inputs X (4 x 2) and targets t (0 or 1)
X4 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
AND = np.array([0, 0, 0, 1])
OR  = np.array([0, 1, 1, 1])
XOR = np.array([0, 1, 1, 0])
print("ready")

## 1 · The perceptron's prediction
A perceptron with weights $w$ (one per input) and bias $b$ computes a **score** and answers 1 if it is positive:

$$\text{output} = \begin{cases} 1 & \text{if } w \cdot x + b > 0 \\ 0 & \text{otherwise} \end{cases}$$

Write `predict` so it works for a whole batch: `X` has one example per row, and you return an array of 0s and 1s.

In [ ]:
def predict(X, w, b):
    # TODO: return an integer array with 1 where X @ w + b > 0, else 0
    raise NotImplementedError("Exercise 1")

**Check 1.** The weights from the lesson's quiz, and a hand-made AND gate.

In [ ]:
assert predict(np.array([[1.0, 1.0]]), np.array([2.0, -1.0]), -0.5).tolist() == [1]
assert predict(X4, np.array([1.0, 1.0]), -1.5).tolist() == AND.tolist(), "w = (1, 1), b = -1.5 is an AND gate"
assert predict(X4, np.array([1.0, 1.0]), -0.5).tolist() == OR.tolist(), "w = (1, 1), b = -0.5 is an OR gate"
assert predict(X4, np.zeros(2), 0.0).tolist() == [0, 0, 0, 0], "a score of exactly 0 is not > 0"
print("✓ exercise 1")

## 2 · Rosenblatt's learning rule
Go through the examples one at a time. When the output $\hat y$ is wrong, nudge the weights:

$$w \leftarrow w + \eta\,(t - \hat y)\,x, \qquad b \leftarrow b + \eta\,(t - \hat y)$$

One pass over all examples is an **epoch**. Stop early as soon as a whole epoch makes no mistakes. Return the final weights, the bias and a list with the number of mistakes in each epoch.

In [ ]:
def train_perceptron(X, t, eta=1.0, epochs=50):
    w = np.zeros(X.shape[1])
    b = 0.0
    mistakes_per_epoch = []
    for epoch in range(epochs):
        mistakes = 0
        for x_i, t_i in zip(X, t):
            # TODO: predict for this one example; if wrong, update w and b and count the mistake
            raise NotImplementedError("Exercise 2")
        mistakes_per_epoch.append(mistakes)
        if mistakes == 0:
            break
    return w, b, mistakes_per_epoch

**Check 2.** AND and OR are linearly separable, so the rule must reach an epoch with 0 mistakes. Then a harder case: 100 random points split by a hidden line.

In [ ]:
for name, t in [("AND", AND), ("OR", OR)]:
    w, b, hist = train_perceptron(X4, t)
    assert hist[-1] == 0, f"{name}: the last epoch should have 0 mistakes, got {hist}"
    assert predict(X4, w, b).tolist() == t.tolist()
    print(f"{name}: converged after {len(hist)} epochs, w = {w}, b = {b}")

# 100 random points, labelled by a line the perceptron doesn't know
Xr = rng.uniform(-1, 1, size=(100, 2))
true_w, true_b = np.array([1.5, -1.0]), 0.2
tr = (Xr @ true_w + true_b > 0).astype(int)
keep = np.abs(Xr @ true_w + true_b) > 0.1          # leave a small gap so it converges quickly
Xr, tr = Xr[keep], tr[keep]
w, b, hist = train_perceptron(Xr, tr, eta=0.1, epochs=200)
assert hist[-1] == 0, "separable data: the perceptron convergence theorem promises 0 mistakes eventually"
print(f"random separable data: 0 mistakes after {len(hist)} epochs")
print("✓ exercise 2")

In [ ]:
# Look at the line it found (optional)
xs = np.linspace(-1, 1, 2)
plt.scatter(Xr[:, 0], Xr[:, 1], c=tr, cmap="coolwarm", s=18)
plt.plot(xs, -(w[0] * xs + b) / w[1], "k-", label="perceptron")
plt.plot(xs, -(true_w[0] * xs + true_b) / true_w[1], "k:", label="true line")
plt.ylim(-1, 1); plt.legend(); plt.title("A learned decision boundary"); plt.show()

## 3 · XOR defeats a single line
Train the same perceptron on XOR. It never reaches an epoch with 0 mistakes.

To show it's not just bad luck, search over many lines: write `best_line_accuracy`, which tries `n` random $(w, b)$ settings and returns the **best** fraction of the four XOR points any of them gets right. Minsky and Papert's point is that this can never reach 1.0.

In [ ]:
w, b, hist = train_perceptron(X4, XOR, epochs=100)
print("mistakes in the last 5 epochs:", hist[-5:])

def best_line_accuracy(X, t, n=20000, seed=0):
    r = np.random.default_rng(seed)
    # TODO: draw n random weight vectors (n x 2) and biases (n,) from a normal distribution,
    #       compute each one's accuracy on (X, t) and return the best accuracy (a float)
    raise NotImplementedError("Exercise 3")

**Check 3**

In [ ]:
assert min(hist) > 0, "the perceptron should never get a mistake-free epoch on XOR"
assert len(hist) == 100, "it should run out of epochs"
acc = best_line_accuracy(X4, XOR)
print("best accuracy of any random line on XOR:", acc)
assert abs(acc - 0.75) < 1e-9, "the best a single line can do on XOR is 3 of 4"
assert best_line_accuracy(X4, AND) == 1.0, "but some line gets AND perfectly"
print("✓ exercise 3")

## 4 · A hidden layer, trained with backprop
Now the 1986 fix: a network with a **hidden layer** of smooth units, trained by **backpropagation** and gradient descent.

$$h = \tanh(X W_1 + b_1), \qquad p = \sigma(h\,W_2 + b_2), \qquad L = -\frac{1}{N}\sum \big[t \log p + (1 - t)\log(1 - p)\big]$$

$\sigma$ is the sigmoid $1/(1 + e^{-z})$, so $p$ is a probability, and $L$ is the cross-entropy loss. Backprop runs the chain rule from the loss back to every weight. For this loss and the sigmoid, the gradient at the output unit's input is neatly $p - t$ (divided by $N$, because $L$ is a mean). After that, each step is **upstream gradient × local derivative**; for $\tanh$ the local derivative is $1 - h^2$.

The forward pass, the training loop and the update are written for you. **Fill in `backward`.**
Shapes: `X` is (N, 2), `W1` (2, H), `b1` (H,), `W2` (H, 1), `b2` (1,), `h` (N, H), `p` and `T` (N, 1).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def forward(X, params):
    W1, b1, W2, b2 = params
    h = np.tanh(X @ W1 + b1)
    p = sigmoid(h @ W2 + b2)
    return h, p

def loss_fn(params, X, T):
    _, p = forward(X, params)
    return -np.mean(T * np.log(p + 1e-12) + (1 - T) * np.log(1 - p + 1e-12))

def train_mlp(X, t, hidden=4, lr=1.0, steps=3000, seed=1):
    r = np.random.default_rng(seed)
    params = [r.normal(0, 1, size=(2, hidden)), np.zeros(hidden), r.normal(0, 1, size=(hidden, 1)), np.zeros(1)]
    T = t.reshape(-1, 1).astype(float)
    losses = []
    for step in range(steps):
        h, p = forward(X, params)
        losses.append(loss_fn(params, X, T))
        grads = backward(X, T, params, h, p)
        params = [P - lr * G for P, G in zip(params, grads)]   # gradient descent
    return params, losses

def mlp_predict(X, params):
    return (forward(X, params)[1].ravel() > 0.5).astype(int)

In [ ]:
def backward(X, T, params, h, p):
    W1, b1, W2, b2 = params
    N = len(X)
    # TODO: dz2 = (p - T) / N, then dW2, db2, dh, dz1 (through tanh: 1 - h**2), dW1, db1
    raise NotImplementedError("Exercise 4")
    return dW1, db1, dW2, db2

**Check 4.** First compare your gradients with numerical ones (nudge each weight by a tiny $\varepsilon$ both ways), then train on XOR.

In [ ]:
r = np.random.default_rng(3)
P0 = [r.normal(0, 1, (2, 4)), r.normal(0, 0.5, 4), r.normal(0, 1, (4, 1)), r.normal(0, 0.5, 1)]
T = XOR.reshape(-1, 1).astype(float)
h, p = forward(X4, P0)
analytic = backward(X4, T, P0, h, p)
for k, G in enumerate(analytic):
    num = np.zeros_like(P0[k])
    for idx in np.ndindex(P0[k].shape):
        up = [a.copy() for a in P0]; up[k][idx] += 1e-6
        dn = [a.copy() for a in P0]; dn[k][idx] -= 1e-6
        num[idx] = (loss_fn(up, X4, T) - loss_fn(dn, X4, T)) / 2e-6
    assert G.shape == P0[k].shape, f"gradient {k} has shape {G.shape}, expected {P0[k].shape}"
    assert np.allclose(G, num, atol=1e-6), (k, G, num)
print("gradient check passed")

params, losses = train_mlp(X4, XOR)
print("loss: start %.3f → end %.4f" % (losses[0], losses[-1]))
assert mlp_predict(X4, params).tolist() == XOR.tolist(), "the network should get all four XOR points right"
assert losses[-1] < 0.05
print("✓ exercise 4: a hidden layer + backprop solves XOR")

In [ ]:
# What did it learn? The decision regions (optional)
g = np.linspace(-0.5, 1.5, 200)
GX, GY = np.meshgrid(g, g)
Z = mlp_predict(np.c_[GX.ravel(), GY.ravel()], params).reshape(GX.shape)
plt.contourf(GX, GY, Z, alpha=0.25, cmap="coolwarm")
plt.scatter(X4[:, 0], X4[:, 1], c=XOR, cmap="coolwarm", s=120, edgecolors="k")
plt.title("XOR, solved by one hidden layer"); plt.show()
plt.plot(losses); plt.yscale("log"); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

## Experiment
1. Set `hidden=1` in `train_mlp(X4, XOR, hidden=1)`. Can one hidden unit solve XOR? Why not? *(Hint: what shape is its decision boundary?)*
2. With `hidden=2`, try seeds 0 to 19. Some runs get stuck. How many? Why might extra hidden units make training more reliable?
3. Thought experiment: if the hidden units used Rosenblatt's hard step (output 0 or 1) instead of `tanh`, the step's slope is 0 everywhere except at the jump. What would backprop's gradient for `W1` be? Why did backprop need *smooth* units?

In [ ]:
# scratch space for the experiments
for H in [1, 2, 4]:
    solved = sum(mlp_predict(X4, train_mlp(X4, XOR, hidden=H, seed=s)[0]).tolist() == XOR.tolist() for s in range(20))
    print(f"hidden={H}: solved XOR in {solved}/20 seeds")

## Recap
- A perceptron draws **one straight line**: $w \cdot x + b = 0$. Its rule (1958) nudges the line after each mistake and is guaranteed to converge when a separating line exists.
- **XOR** has no separating line, so no amount of training helps: the best is 3 of 4 (Minsky & Papert, 1969).
- A **hidden layer** of smooth units can bend the boundary, and **backpropagation** (popularised in 1986) computes the gradients to train it. That recipe still trains every LLM.

Back to the course and hit **Mark complete** to add the review cards.

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def predict(X, w, b):
    return (X @ w + b > 0).astype(int)

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def train_perceptron(X, t, eta=1.0, epochs=50):
    w = np.zeros(X.shape[1])
    b = 0.0
    mistakes_per_epoch = []
    for epoch in range(epochs):
        mistakes = 0
        for x_i, t_i in zip(X, t):
            y_hat = 1 if x_i @ w + b > 0 else 0
            if y_hat != t_i:
                w = w + eta * (t_i - y_hat) * x_i
                b = b + eta * (t_i - y_hat)
                mistakes += 1
        mistakes_per_epoch.append(mistakes)
        if mistakes == 0:
            break
    return w, b, mistakes_per_epoch

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def best_line_accuracy(X, t, n=20000, seed=0):
    r = np.random.default_rng(seed)
    W = r.normal(size=(n, 2))
    B = r.normal(size=n)
    preds = (X @ W.T + B > 0).astype(int)          # (4, n): every line's answers
    acc = (preds == t[:, None]).mean(axis=0)        # accuracy of each line
    return float(acc.max())

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def backward(X, T, params, h, p):
    W1, b1, W2, b2 = params
    N = len(X)
    dz2 = (p - T) / N                 # (N, 1)
    dW2 = h.T @ dz2                   # (H, 1)
    db2 = dz2.sum(axis=0)             # (1,)
    dh = dz2 @ W2.T                   # (N, H)
    dz1 = dh * (1 - h ** 2)           # back through tanh
    dW1 = X.T @ dz1                   # (2, H)
    db1 = dz1.sum(axis=0)             # (H,)
    return dW1, db1, dW2, db2